# Run a calibration

Run `qdash config init --project-id your-project-id` in a terminal, then fill in `api_token` in the generated file. Use `--base-url` for a remote deployment. If the file already exists, edit it instead. `QDash()` reads its `default_profile`; use `QDash.from_profile("local")` to select a profile explicitly. See the [Python reference](../../../docs/reference/clients/python.md#connect-to-a-deployment) for setup. Replace the chip ID and choose settings suitable for your hardware. A calibration session is a named objective on the chip that groups the runs started for it; it holds no hardware and is not a connection.

In [ ]:
from qdash.calibration import CustomOneQubit, QubitTargets, TaskCall, sweep
from qdash.sdk import CalibrationRunError, CalibrationSession, QDash

client = QDash()
session = CalibrationSession.create(client, chip_id="YOUR_CHIP_ID", name="Coherence check")
print("Session:", session.session_id)

## Define Calibration

In [ ]:
targets = QubitTargets(["0", "1"])
steps = [
    CustomOneQubit(
        step_name="t1",
        tasks=[
            TaskCall(name="Configure"),
            TaskCall(name="CheckT1", run={"time_range": sweep.linspace(300, 500_000, 51)}),
        ],
    ),
]
run_parameters = {"readout_duration": 1024, "interval": 262_144}

## Validate Calibration

Validation does not start a measurement. Review any problems before continuing.

In [ ]:
check = session.validate(targets, steps, default_run_parameters=run_parameters)
print("Valid:", check.valid)
for problem in check.problems:
    print(problem)
for warning in check.warnings:
    print("Warning:", warning)

## Run Calibration

Running this cell starts a measurement in the session and waits for the whole run; re-running it starts another run in the same session. A failed or cancelled run raises `CalibrationRunError`. A wait timeout does not cancel the run: use `session.submit(...)` to start one without waiting and follow it with `launch.wait()`.

In [ ]:
try:
    result = session.run(targets, steps, default_run_parameters=run_parameters, timeout_seconds=600)
except CalibrationRunError as error:
    print("Run ended:", error.result.status)
else:
    print(result.status, result.execution_ids)

## Close Session and Connection

Closing the session ends the objective; it stops nothing. Closing the client ends the connection.

In [ ]:
session.close(note="Coherence check done")